<a href="https://colab.research.google.com/github/waltersalles/Conversas_Voz_IA_Gemini/blob/main/Conversas_Voz_IA_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install SpeechRecognition pydub google-generativeai gtts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.2
    Uninstalling click-8.3.2:
      Successfully uninstalled click-8.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [4]:
import google.generativeai as genai
from google.colab import userdata

# Configura o Gemini
genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))
model = genai.GenerativeModel('gemini-1.5-flash')


In [8]:
from IPython.display import HTML, Audio, display
from google.colab.output import eval_js
from base64 import b64decode

# Código JavaScript para capturar o microfone pelo navegador
VIDEO_HTML = """
<script>
var my_recorder;
var my_stream;

async function startRecording() {
  my_stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  my_recorder = new MediaRecorder(my_stream);
  var chunks = [];
  my_recorder.ondataavailable = (e) => chunks.push(e.data);
  my_recorder.start();
  return new Promise(resolve => {
    my_recorder.onstop = async () => {
      var blob = new Blob(chunks);
      var reader = new FileReader();
      reader.readAsDataURL(blob);
      reader.onloadend = () => resolve(reader.result);
    };
  });
}

function stopRecording() {
  my_recorder.stop();
  my_stream.getTracks().forEach(i => i.stop());
}
</script>
<div style="border: 1px solid #ccc; padding: 10px; border-radius: 10px; width: fit-content; background: #f9f9f9;">
  <p style="margin-top: 0;">🎤 <b>Gravador de Voz para o Desafio</b></p>
  <button onclick="startRecording()" style="background: #d9534f; color: white; border: none; padding: 5px 15px; border-radius: 5px; cursor: pointer;">🔴 Gravar</button>
  <button onclick="stopRecording()" style="background: #5bc0de; color: white; border: none; padding: 5px 15px; border-radius: 5px; cursor: pointer;">⏹️ Parar</button>
</div>
"""

def record_audio(filename='input_audio.wav'):
    display(HTML(VIDEO_HTML))
    data = eval_js("startRecording()")

    # Processa o áudio quando você clica em Parar
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    print(f"✅ Áudio salvo com sucesso como: {filename}")

# Inicia o processo
record_audio()

✅ Áudio salvo com sucesso como: input_audio.wav


In [10]:
import speech_recognition as sr
from pydub import AudioSegment

# 1. Converte o áudio gravado (WebM/Outros) para WAV puro
try:
    audio_original = AudioSegment.from_file("input_audio.wav")
    audio_original.export("input_audio_fixed.wav", format="wav")
    print("✅ Áudio convertido para WAV com sucesso!")
except Exception as e:
    print(f"Erro na conversão: {e}. Verifique se gravou o áudio corretamente.")

# 2. Agora fazemos a transcrição usando o arquivo corrigido
r = sr.Recognizer()
with sr.AudioFile("input_audio_fixed.wav") as source:
    audio_data = r.record(source)
    try:
        # Usa o motor gratuito do Google para transcrever
        texto_pergunta = r.recognize_google(audio_data, language="pt-BR")
        print(f"🎤 Você disse: {texto_pergunta}")
        print("-" * 30)

        # 3. Envia para o Gemini responder (Cérebro)
        print("🤖 IA pensando...")
        response = model.generate_content(texto_pergunta)
        texto_resposta = response.text
        print(f"💬 Resposta da IA: {texto_resposta}")

    except sr.UnknownValueError:
        print("A IA não conseguiu entender o áudio. Tente falar mais claro ou mais perto do microfone.")
    except Exception as e:
        print(f"Erro no processamento: {e}")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


✅ Áudio convertido para WAV com sucesso!
A IA não conseguiu entender o áudio. Tente falar mais claro ou mais perto do microfone.


In [12]:
import speech_recognition as sr
from pydub import AudioSegment
from gtts import gTTS
from IPython.display import Audio, display

# 1. Garante a conversão do áudio gravado
try:
    audio_original = AudioSegment.from_file("input_audio.wav")
    audio_original.export("input_audio_fixed.wav", format="wav")

    # 2. Transcrição (Ouvido)
    r = sr.Recognizer()
    with sr.AudioFile("input_audio_fixed.wav") as source:
        audio_data = r.record(source)
        texto_pergunta = r.recognize_google(audio_data, language="pt-BR")
        print(f"🎤 Você disse: {texto_pergunta}")

    # 3. Inteligência (Cérebro Gemini)
    print("🤖 IA pensando...")
    response = model.generate_content(texto_pergunta)
    texto_final = response.text # Usando um nome fixo aqui
    print(f"💬 Resposta da IA: {texto_final}")

    # 4. Síntese de Voz (Saída)
    print("🔊 Gerando áudio da resposta...")
    tts = gTTS(text=texto_final, lang='pt')
    tts.save("resposta_final.mp3")

    # Toca o áudio automaticamente
    display(Audio("resposta_final.mp3", autoplay=True))

except Exception as e:
    print(f"❌ Ocorreu um erro: {e}")
    print("Dica: Verifique se você gravou o áudio no passo anterior!")

❌ Ocorreu um erro: 
Dica: Verifique se você gravou o áudio no passo anterior!
